
### Cross-Validation (200 iters) for allomertic model:
    # y = a0 * x1^a1 * x2^a2 * x3^a3
# Target variable: Foliage_t_ha
# Inputs: DBH_cm, H_m, BA_sq_m_sq_m


In [1]:
# Import libraries

import os
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.stats import t as student_t

/home/dima/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/dima/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
# Config

PATH = "./../01_input_data/input_preparated_biomass_data_for_analisys_20230125.xlsx"
Y_COL = "Foliage_t_ha"
X_COLS = ["DBH_cm", "H_m", "BA_sq_m_sq_m"]
TEST_SIZE = 0.2
N_ITER = 200
A_BOUNDS = (-6.0, 6.0)      # bounds for exponents a_j
X_FLOOR = 1e-6              # small floor to allow negative exponents
ROBUST_LOSS = "linear"      # LS loss; could switch to "soft_l1" for extra robustness
RANDOM_SEED = 4242
rng = np.random.default_rng(RANDOM_SEED)

In [3]:
# Output config

OUT_DIR = "./model_FOLIAGE"
os.makedirs(OUT_DIR, exist_ok=True)

PER_SPECIES_CSV = os.path.join(OUT_DIR, "FOLIAGE_cv200_results_SE.csv")
SUMMARY_PARAMS_QEXT_CSV = os.path.join(OUT_DIR, "FOLIAGE_cv200_summary_params_SE.csv")
OVERALL_METRICS_CSV = os.path.join(OUT_DIR, "FOLIAGE_cv200_overall_metrics.csv")
SUMMARY_OVERALL_QEXT_CSV = os.path.join(OUT_DIR, "FOLIAGE_cv200_summary_overall_metrics.csv")


In [4]:
# Clean outputs if re-running
for p in (PER_SPECIES_CSV, SUMMARY_PARAMS_QEXT_CSV, OVERALL_METRICS_CSV, SUMMARY_OVERALL_QEXT_CSV):
    try:
        os.remove(p)
    except FileNotFoundError:
        pass

In [5]:
# ----------------- Load & prep -----------------
df0 = pd.read_excel(PATH)

needed = ["Species", Y_COL] + X_COLS
missing_cols = [c for c in needed if c not in df0.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df0[needed].copy()
for c in [Y_COL] + X_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Species", Y_COL] + X_COLS).copy()

In [6]:
# Positive floor for predictors (no logs; stability for negative exponents)
for c in X_COLS:
    df[c] = np.maximum(df[c].to_numpy(float), X_FLOOR)

# Ensure y > 0
df = df[df[Y_COL] > 0].copy()

species_list = sorted(df["Species"].astype(str).unique())
k = len(X_COLS)
p = k + 1  # number of parameters

# Modeling elements

In [7]:
def _yhat(theta, X):
    a0 = theta[0]; a = theta[1:]
    P = np.ones(X.shape[0], dtype=float)
    for j in range(X.shape[1]):
        P *= np.power(X[:, j], a[j])
    return a0 * P

def _res(theta, X, y):
    return _yhat(theta, X) - y

def _jac(theta, X):
    a0 = theta[0]; a = theta[1:]
    n, k_ = X.shape
    P = np.ones(n, dtype=float)
    lX = np.log(X)
    for j in range(k_):
        P *= np.power(X[:, j], a[j])
    yhat = a0 * P
    J = np.empty((n, k_+1), dtype=float)
    J[:, 0] = P
    for j in range(k_):
        J[:, j+1] = yhat * lX[:, j]
    return J

def fit_with_se(Xtr, ytr, a_bounds=A_BOUNDS):
    """Fit on training data; return params, SEs, and 95% CI using t-dist with df=n-p."""
    n = Xtr.shape[0]
    low = np.r_[1e-12, np.full(k, a_bounds[0])]
    high = np.r_[np.inf,  np.full(k, a_bounds[1])]
    a0_start = float(np.clip(np.median(ytr), 1e-3, np.inf))
    p0 = np.r_[a0_start, np.zeros(k)]
    res = least_squares(
        fun=lambda th: _res(th, Xtr, ytr),
        x0=p0,
        jac=lambda th: _jac(th, Xtr),
        method="trf",
        bounds=(low, high),
        loss=ROBUST_LOSS,
        max_nfev=5000
    )
    params = res.x
    # Covariance-based SE and 95% CI
    try:
        J = _jac(params, Xtr)
        r = _res(params, Xtr, ytr)
        RSS = float(np.dot(r, r))
        dof = max(n - p, 1)
        sigma2 = RSS / dof
        JTJ_inv = np.linalg.pinv(J.T @ J)
        cov = sigma2 * JTJ_inv
        SE = np.sqrt(np.clip(np.diag(cov), 0, np.inf))
        tcrit = float(student_t.ppf(0.975, dof)) if dof > 1 else 1.96
        CI_lo = params - tcrit * SE
        CI_hi = params + tcrit * SE
    except Exception:
        SE = np.full_like(params, np.nan)
        CI_lo = np.full_like(params, np.nan)
        CI_hi = np.full_like(params, np.nan)

    return res, params, SE, CI_lo, CI_hi

def r2(y_true, y_pred):
    rss = float(np.sum((y_true - y_pred)**2))
    tss = float(np.sum((y_true - np.mean(y_true))**2))
    return 1 - rss/tss if tss > 0 else np.nan

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred)**2))) if len(y_true) else np.nan

def bias(y_true, y_pred):
    return float(np.mean(y_pred - y_true)) if len(y_true) else np.nan

def rel(x, denom):
    return float(x / denom) if denom not in (0, np.nan) and np.isfinite(denom) else np.nan

def stratified_split(df_in, test_size=TEST_SIZE):
    train_parts, test_parts = [], []
    for species, g in df_in.groupby("Species"):
        # shuffle with RNG for reproducibility across iters
        g = g.sample(frac=1.0, random_state=int(rng.integers(0, 1_000_000)))
        n = len(g)
        if n <= 1:
            n_test = 0
        else:
            n_test = int(round(test_size * n))
            if n >= 5 and n_test == 0: n_test = 1
            if n_test >= n: n_test = n - 1
        test_g = g.iloc[:n_test].copy()
        train_g = g.iloc[n_test:].copy()
        if len(train_g): train_parts.append(train_g)
        if len(test_g):  test_parts.append(test_g)
    train_df = pd.concat(train_parts, ignore_index=True) if len(train_parts) else df_in.iloc[0:0].copy()
    test_df  = pd.concat(test_parts,  ignore_index=True) if len(test_parts)  else df_in.iloc[0:0].copy()
    return train_df, test_df

In [8]:
# Iteration loops
def run_species_chunk(start_iter, end_iter):
    rows = []
    for it in range(start_iter, end_iter+1):
        train_df, test_df = stratified_split(df, TEST_SIZE)
        for sp in species_list:
            tr = train_df[train_df["Species"].astype(str) == sp]
            te = test_df[test_df["Species"].astype(str) == sp]
            n_tr, n_te = len(tr), len(te)
            if n_tr >= p and n_te >= 1:
                Xtr = tr[X_COLS].to_numpy(float)
                ytr = tr[Y_COL].to_numpy(float)
                try:
                    res, params, SE, CI_lo, CI_hi = fit_with_se(Xtr, ytr)
                    row = {
                        "iter": it, "Species": sp, "n_train": n_tr, "n_test": n_te,
                        "a0": float(params[0]),
                        f"a_{X_COLS[0]}": float(params[1]),
                        f"a_{X_COLS[1]}": float(params[2]),
                        f"a_{X_COLS[2]}": float(params[3]),
                        "SE_a0": float(SE[0]),
                        f"SE_a_{X_COLS[0]}": float(SE[1]),
                        f"SE_a_{X_COLS[1]}": float(SE[2]),
                        f"SE_a_{X_COLS[2]}": float(SE[3]),
                        "CI95_lo_a0": float(CI_lo[0]), "CI95_hi_a0": float(CI_hi[0]),
                        f"CI95_lo_a_{X_COLS[0]}": float(CI_lo[1]), f"CI95_hi_a_{X_COLS[0]}": float(CI_hi[1]),
                        f"CI95_lo_a_{X_COLS[1]}": float(CI_lo[2]), f"CI95_hi_a_{X_COLS[1]}": float(CI_hi[2]),
                        f"CI95_lo_a_{X_COLS[2]}": float(CI_lo[3]), f"CI95_hi_a_{X_COLS[2]}": float(CI_hi[3]),
                        "success": bool(res.success)
                    }
                except Exception:
                    row = {
                        "iter": it, "Species": sp, "n_train": n_tr, "n_test": n_te,
                        "a0": np.nan,
                        f"a_{X_COLS[0]}": np.nan,
                        f"a_{X_COLS[1]}": np.nan,
                        f"a_{X_COLS[2]}": np.nan,
                        "SE_a0": np.nan,
                        f"SE_a_{X_COLS[0]}": np.nan,
                        f"SE_a_{X_COLS[1]}": np.nan,
                        f"SE_a_{X_COLS[2]}": np.nan,
                        "CI95_lo_a0": np.nan, "CI95_hi_a0": np.nan,
                        f"CI95_lo_a_{X_COLS[0]}": np.nan, f"CI95_hi_a_{X_COLS[0]}": np.nan,
                        f"CI95_lo_a_{X_COLS[1]}": np.nan, f"CI95_hi_a_{X_COLS[1]}": np.nan,
                        f"CI95_lo_a_{X_COLS[2]}": np.nan, f"CI95_hi_a_{X_COLS[2]}": np.nan,
                        "success": False
                    }
            else:
                row = {
                    "iter": it, "Species": sp, "n_train": n_tr, "n_test": n_te,
                    "a0": np.nan,
                    f"a_{X_COLS[0]}": np.nan,
                    f"a_{X_COLS[1]}": np.nan,
                    f"a_{X_COLS[2]}": np.nan,
                    "SE_a0": np.nan,
                    f"SE_a_{X_COLS[0]}": np.nan,
                    f"SE_a_{X_COLS[1]}": np.nan,
                    f"SE_a_{X_COLS[2]}": np.nan,
                    "CI95_lo_a0": np.nan, "CI95_hi_a0": np.nan,
                    f"CI95_lo_a_{X_COLS[0]}": np.nan, f"CI95_hi_a_{X_COLS[0]}": np.nan,
                    f"CI95_lo_a_{X_COLS[1]}": np.nan, f"CI95_hi_a_{X_COLS[1]}": np.nan,
                    f"CI95_lo_a_{X_COLS[2]}": np.nan, f"CI95_hi_a_{X_COLS[2]}": np.nan,
                    "success": False
                }
            rows.append(row)
    chunk_df = pd.DataFrame(rows)
    if os.path.exists(PER_SPECIES_CSV):
        prev = pd.read_csv(PER_SPECIES_CSV)
        out = pd.concat([prev, chunk_df], ignore_index=True)
        out.to_csv(PER_SPECIES_CSV, index=False)
    else:
        chunk_df.to_csv(PER_SPECIES_CSV, index=False)
    return {"rows": len(rows)}

def run_overall_chunk(start_iter, end_iter):
    rows = []
    for it in range(start_iter, end_iter+1):
        train_df, test_df = stratified_split(df, TEST_SIZE)
        ytr_all, ytrhat_all = [], []
        yte_all, ytehat_all = [], []
        for sp, tr in train_df.groupby(train_df["Species"].astype(str)):
            te = test_df[test_df["Species"].astype(str) == sp]
            if len(tr) >= p and len(te) >= 1:
                Xtr = tr[X_COLS].to_numpy(float); ytr = tr[Y_COL].to_numpy(float)
                Xte = te[X_COLS].to_numpy(float); yte = te[Y_COL].to_numpy(float)
                try:
                    res, params, *_ = fit_with_se(Xtr, ytr)
                    ytr_hat = _yhat(params, Xtr)
                    yte_hat = _yhat(params, Xte)
                    ytr_all.append(ytr);     ytrhat_all.append(ytr_hat)
                    yte_all.append(yte);     ytehat_all.append(yte_hat)
                except Exception:
                    pass
        if ytr_all and yte_all:
            ytr_all = np.concatenate(ytr_all); ytrhat_all = np.concatenate(ytrhat_all)
            yte_all = np.concatenate(yte_all); ytehat_all = np.concatenate(ytehat_all)
            # Metrics
            rss_tr = float(np.sum((ytr_all - ytrhat_all)**2))
            tss_tr = float(np.sum((ytr_all - np.mean(ytr_all))**2))
            R2_tr = 1 - rss_tr/tss_tr if tss_tr>0 else np.nan
            RMSE_tr = float(np.sqrt(np.mean((ytr_all - ytrhat_all)**2)))
            rss_te = float(np.sum((yte_all - ytehat_all)**2))
            tss_te = float(np.sum((yte_all - np.mean(yte_all))**2))
            R2_te = 1 - rss_te/tss_te if tss_te>0 else np.nan
            RMSE_te = float(np.sqrt(np.mean((yte_all - ytehat_all)**2)))
            R2_tr_pct = 100.0 * R2_tr if np.isfinite(R2_tr) else np.nan
            R2_te_pct = 100.0 * R2_te if np.isfinite(R2_te) else np.nan
            mean_tr = float(np.mean(ytr_all))
            mean_te = float(np.mean(yte_all))
            rRMSE_tr = float(RMSE_tr/mean_tr) if mean_tr!=0 else np.nan
            rRMSE_te = float(RMSE_te/mean_te) if mean_te!=0 else np.nan
            B_tr = float(np.mean(ytrhat_all - ytr_all))
            B_te = float(np.mean(ytehat_all - yte_all))
            rBias_tr = float(B_tr/mean_tr) if mean_tr!=0 else np.nan
            rBias_te = float(B_te/mean_te) if mean_te!=0 else np.nan
            PBIAS_tr = 100.0 * float(np.sum(ytrhat_all - ytr_all) / np.sum(ytr_all)) if np.sum(ytr_all)!=0 else np.nan
            PBIAS_te = 100.0 * float(np.sum(ytehat_all - yte_all) / np.sum(yte_all)) if np.sum(yte_all)!=0 else np.nan

            rows.append({
                "iter": it,
                "R2_train_all": R2_tr, "R2_train_all_pct": R2_tr_pct,
                "RMSE_train_all": RMSE_tr, "rRMSE_train_all": rRMSE_tr,
                "Bias_train_all": B_tr, "rBias_train_all": rBias_tr, "PBIAS_train_all_pct": PBIAS_tr,
                "R2_test_all": R2_te, "R2_test_all_pct": R2_te_pct,
                "RMSE_test_all": RMSE_te, "rRMSE_test_all": rRMSE_te,
                "Bias_test_all": B_te, "rBias_test_all": rBias_te, "PBIAS_test_all_pct": PBIAS_te,
                "n_train_all": int(len(ytr_all)), "n_test_all": int(len(yte_all))
            })
        else:
            rows.append({
                "iter": it,
                "R2_train_all": np.nan, "R2_train_all_pct": np.nan,
                "RMSE_train_all": np.nan, "rRMSE_train_all": np.nan,
                "Bias_train_all": np.nan, "rBias_train_all": np.nan, "PBIAS_train_all_pct": np.nan,
                "R2_test_all": np.nan, "R2_test_all_pct": np.nan,
                "RMSE_test_all": np.nan, "rRMSE_test_all": np.nan,
                "Bias_test_all": np.nan, "rBias_test_all": np.nan, "PBIAS_test_all_pct": np.nan,
                "n_train_all": 0, "n_test_all": 0
            })
    df_overall = pd.DataFrame(rows)
    if os.path.exists(OVERALL_METRICS_CSV):
        prev = pd.read_csv(OVERALL_METRICS_CSV)
        out = pd.concat([prev, df_overall], ignore_index=True)
        out.to_csv(OVERALL_METRICS_CSV, index=False)
    else:
        df_overall.to_csv(OVERALL_METRICS_CSV, index=False)
    return {"rows": len(rows)}

In [9]:
# Summaries for quantelies
def extended_summary(df_in: pd.DataFrame, cols, name_col="name"):
    qs = {
        "q2.5": 0.025,
        "q15.9": 0.159,
        "q25": 0.25,
        "q50": 0.50,
        "q75": 0.75,
        "q84.1": 0.841,
        "q97.5": 0.975,
    }
    out = {
        "count": df_in[cols].count(),
        "mean":  df_in[cols].mean(),
        "std":   df_in[cols].std(),
        "min":   df_in[cols].min(),
        "max":   df_in[cols].max(),
    }
    for name, qv in qs.items():
        out[name] = df_in[cols].quantile(qv)
    return pd.DataFrame(out).reset_index(names=name_col)

def build_summaries():
    res_species = pd.read_csv(PER_SPECIES_CSV)
    overall_df  = pd.read_csv(OVERALL_METRICS_CSV)

    # Per-species blocks (params, SE, CI)
    param_cols = [c for c in res_species.columns if c == "a0" or c.startswith("a_")]
    se_cols    = [c for c in res_species.columns if c.startswith("SE_")]
    ci_lo_cols = [c for c in res_species.columns if c.startswith("CI95_lo_")]
    ci_hi_cols = [c for c in res_species.columns if c.startswith("CI95_hi_")]

    summary_blocks = []
    for sp, g in res_species.groupby("Species"):
        g_ok = g[g["success"] == True]
        if len(g_ok) == 0:
            continue
        s_param = extended_summary(g_ok, param_cols, "parameter"); s_param.insert(0, "Species", sp); s_param.insert(1, "block", "param")
        s_se    = extended_summary(g_ok, se_cols, "parameter");   s_se.insert(0, "Species", sp);    s_se.insert(1, "block", "SE")
        s_cilo  = extended_summary(g_ok, ci_lo_cols, "parameter");s_cilo.insert(0, "Species", sp);  s_cilo.insert(1, "block", "CI95_lo")
        s_cihi  = extended_summary(g_ok, ci_hi_cols, "parameter");s_cihi.insert(0, "Species", sp);  s_cihi.insert(1, "block", "CI95_hi")
        summary_blocks += [s_param, s_se, s_cilo, s_cihi]

    summary_params_qext = pd.concat(summary_blocks, ignore_index=True) if summary_blocks else pd.DataFrame()
    summary_params_qext.to_csv(SUMMARY_PARAMS_QEXT_CSV, index=False)

    # Overall metrics summary with extra quantiles
    metric_cols = [c for c in overall_df.columns if c != "iter"]
    overall_summary_qext = extended_summary(overall_df, metric_cols, "metric")
    overall_summary_qext.to_csv(SUMMARY_OVERALL_QEXT_CSV, index=False)

In [10]:
# Main

# Species-wise params with SE/CI
run_species_chunk(1, 200)

# Overall per-iteration metrics
run_overall_chunk(1, 200)

# Build summaries 
build_summaries()

print("Done. Files saved to:", OUT_DIR)

Done. Files saved to: ./model_FOLIAGE
